# Conformal Prediction (MC-Dropout-normalized) under 30% missingness -- uncapped RUL

In [1]:
import pandas as pd
import numpy as np

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


def create_masked_dataset(df, features):

    data = df.copy()
    mask_features = []

    for feature in features:
        mask_name = feature + "_mask"
        data[mask_name] = data[feature].notna().astype("float32")
        mask_features.append(mask_name)

    data[features] = data[features].fillna(0.0).astype("float32")

    return data, mask_features


In [2]:
train_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone\Dataset\Missing_Train\processed_train_missing_30.csv")
test_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone\Dataset\Missing_Test\processed_test_missing_30.csv")

# Swap in the true uncapped RUL -- the file above only has the capped
# version baked in, inherited from the capped processed_train.csv it was
# generated from. Engine_ID + Cycle uniquely identify a row and are
# untouched by the sensor masking, so they are a safe merge key.
train_uncapped = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\processed_train_uncapped.csv")[["Engine_ID", "Cycle", "RUL"]]

train_missing = train_missing.drop(columns=["RUL"]).merge(
    train_uncapped,
    on=["Engine_ID", "Cycle"],
    how="left"
)

assert train_missing["RUL"].isna().sum() == 0, "merge dropped rows -- Engine_ID/Cycle mismatch"
print("Uncapped RUL range (train):", train_missing["RUL"].min(), "-", train_missing["RUL"].max())

train_masked, MASK_FEATURES = create_masked_dataset(train_missing, FEATURES)
test_masked, _ = create_masked_dataset(test_missing, FEATURES)

MODEL_FEATURES = FEATURES + MASK_FEATURES
print("Model features (raw + mask):", len(MODEL_FEATURES))

np.random.seed(42)

engine_ids = train_masked["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = train_masked[train_masked["Engine_ID"].isin(train_ids)].copy()
val_df = train_masked[train_masked["Engine_ID"].isin(val_ids)].copy()
test_df = train_masked[train_masked["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Uncapped RUL range (train): 0 - 361
Model features (raw + mask): 34
Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [4]:
X_train, y_train = create_sequences(train_df, MODEL_FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, MODEL_FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, MODEL_FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 34)
(2720, 30, 34)
(2695, 30, 34)


## Train the point-prediction model (input width = 34)

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

bilstm = Sequential([
    Input(shape=(WINDOW, len(MODEL_FEATURES))),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

bilstm.summary()


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        50,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,017 (367.25 KB)

 Trainable params: 94,017 (367.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)

history_bilstm = bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - loss: 7165.5430 - mae: 66.6017 - val_loss: 3850.2800 - val_mae: 50.4377 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 3849.7900 - mae: 50.1991 - val_loss: 3652.9834 - val_mae: 50.1817 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 3828.9067 - mae: 50.3135 - val_loss: 3655.3901 - val_mae: 50.2211 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 3825.6331 - mae: 50.2882 - val_loss: 3655.8357 - val_mae: 50.2279 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 3829.1843 - mae: 50.3225 - val_loss: 3654.7913 - val_mae: 50.2126 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 3827.1404 - mae: 50.3392 - val_loss: 3654.7949 - val_mae: 50.2130 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 3823.8083 - mae: 50.3102 

## Per-point difficulty signal (MC Dropout)

In [7]:
def mc_dropout_predict(model, X, n_passes=100):
    predictions = []
    for _ in range(n_passes):
        pred = model(X, training=True).numpy().ravel()
        predictions.append(pred)
    predictions = np.array(predictions)
    return predictions.mean(axis=0), predictions.std(axis=0)


## Calibrate on `val_df`

`ALPHA = 0.10` targets 90% coverage, same as every other conformal notebook in this project.

In [8]:
val_mean, val_std = mc_dropout_predict(bilstm, X_val, n_passes=100)

EPS = 1e-6
nonconformity = np.abs(y_val - val_mean) / (val_std + EPS)

ALPHA = 0.10  # target 90% coverage
q = np.quantile(nonconformity, 1 - ALPHA)

print("Conformal quantile (q):", q)


Conformal quantile (q): 6.292547


In [9]:
cal_mean, cal_std = mc_dropout_predict(bilstm, X_test, n_passes=100)
cal_interval_width = q * cal_std

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, cal_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, cal_mean)))
print("R\u00b2  :", r2_score(y_test, cal_mean))

coverage = (np.abs(y_test - cal_mean) <= cal_interval_width).mean()
print(f"\nEmpirical coverage on held-out split: {coverage:.1%} (target: {1 - ALPHA:.0%})")


Held-out split:
MAE : 29.903728485107422
RMSE: 41.177936282959074
R²  : 0.5857011675834656

Empirical coverage on held-out split: 86.7% (target: 90%)


## Accept/abstain threshold, calibrated on `val_df`

In [10]:
ACCEPT_RATE = 0.80

val_interval_width = q * val_std
cp_threshold = np.quantile(val_interval_width, ACCEPT_RATE)

print("Conformal interval-width threshold:", cp_threshold)


Conformal interval-width threshold: 62.573826


## Official test set (uncapped, 30% missing)

In [11]:
output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\RUL_FD001.txt"
    ,header=None,
    names=["RUL"]
)
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
# NOTE: left uncapped on purpose -- training labels are uncapped here too.

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

# IMPORTANT: test_masked, not train_masked/train_df -- this is the
# official held-out test set, built from test_missing above.
X_test_final, test_engine_ids = create_test_sequences(test_masked, MODEL_FEATURES, WINDOW)


In [12]:
cp_test_mean, cp_test_std = mc_dropout_predict(bilstm, X_test_final, n_passes=100)
cp_test_interval_width = q * cp_test_std

pred_df = pd.DataFrame({"Engine_ID": test_engine_ids, "Predicted_RUL": cp_test_mean, "IntervalWidth": cp_test_interval_width})
comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
cp_test_mean = comparison["Predicted_RUL"].values
cp_test_interval_width = comparison["IntervalWidth"].values


In [13]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008)."""
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [14]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, cp_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, cp_test_mean)))
print("R\u00b2   :", r2_score(actual, cp_test_mean))
print("Score:", nasa_score(actual, cp_test_mean))

official_coverage = (np.abs(actual - cp_test_mean) <= cp_test_interval_width).mean()
print(f"\nEmpirical coverage on official test set: {official_coverage:.1%} (target: {1 - ALPHA:.0%})")


100% coverage:
MAE  : 19.352336883544922
RMSE : 26.38205599300344
R²   : 0.5969511270523071
Score: 5071.987528027932

Empirical coverage on official test set: 95.0% (target: 90%)


## Accept / abstain sweep

In [15]:
accept = cp_test_interval_width <= cp_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 87
Abstained: 13
Coverage : 0.87


In [16]:
y_true_accept = actual[accept]
y_pred_accept = cp_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 18.44270896911621
RMSE : 25.948909152996563
R²   : 0.5855188369750977
Score: 4697.818805071009


In [17]:
for rate in [0.9, 0.8, 0.7]:
    threshold = np.quantile(val_interval_width, rate)
    accept = cp_test_interval_width <= threshold
    rmse = np.sqrt(mean_squared_error(actual[accept], cp_test_mean[accept]))
    score = nasa_score(actual[accept], cp_test_mean[accept])
    print(f"Acceptance={rate:.0%} | Accepted={accept.sum()} | RMSE={rmse:.3f} | Score={score:.1f}")


Acceptance=90% | Accepted=94 | RMSE=26.019 | Score=4834.5
Acceptance=80% | Accepted=87 | RMSE=25.949 | Score=4697.8
Acceptance=70% | Accepted=78 | RMSE=22.284 | Score=2566.0


In [18]:
import os

METHOD = "conformal"
SCENARIO = "missing_uncapped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = cp_test_mean
uncertainty = cp_test_interval_width

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\conformal__missing_uncapped.csv
